In [1]:
%pip install -r requirements.txt

  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB 960.0 kB/s eta 0:00:05
    --------------------------------------- 0.1/4.0 MB 656.4 kB/s eta 0:00:06
    --------------------------------------- 0.1/4.0 MB 871.5 kB/s eta 0:00:05
   - -------------------------------------- 0.2/4.0 MB 1.0 MB/s eta 0:00:04
   -- ------------------------------------- 0.2/4.0 MB 1.3 MB/s eta 0:00:03
   ---- ----------------------------------- 0.4/4.0 MB 1.7 MB/s eta 0:00:03
   ---- ----------------------------------- 0.5/4.0 MB 1.9 MB/s eta 0:00:02
   -------- ------------------------------- 0.8/4.0 MB 2.7 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 3.0 MB/s eta 0:00:01
   ------------- -------------------------- 1.3/4.0 MB 3.4 MB/s eta 0:00:01
   ------------- -------------------------- 1.3/4.0 MB 3.4 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv # 1. Import the loader
from google import genai
from google.genai import types

# 2. Load the environment variables from your .env file
load_dotenv() 

# Ensure API key is set safely via environment variables
# Note: You can check for either GOOGLE_API_KEY or GEMINI_API_KEY based on your setup
if not os.environ.get("GEMINI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
    raise ValueError("Please set your GEMINI_API_KEY or GOOGLE_API_KEY environment variable.")

# Initialize the GenAI Client
# The SDK will automatically pick up GEMINI_API_KEY or GOOGLE_API_KEY if loaded properly
client = genai.Client()

# ==========================================
# 1. TOOL DEFINITIONS
# ==========================================

def lookup_order(order_id: str) -> str:
    """
    Looks up details for a specific order ID from the local repository.
    Returns item, price, purchase date, and warranty duration in months.
    """
    try:
        with open("orders.json", "r") as f:
            orders = json.load(f)
        
        if order_id in orders:
            return json.dumps(orders[order_id])
        return f"Error: Order {order_id} not found."
    except Exception:
        return "Error: Database currently unavailable."

def calculate(expression: str) -> str:
    """
    Evaluates a simple arithmetic expression string (e.g., '89.99 * 2').
    """
    try:
        # Using a restricted/safe evaluation strategy for simple math expressions
        allowed_chars = "0123456789+-*/.() "
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in arithmetic expression."
        
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

# ==========================================
# 2. AGENT CONFIGURATION & INSTRUCTIONS
# ==========================================

agent_instructions = (
    "You are a helpful customer service orders assistant. "
    f"Today's date is strictly {datetime.now().strftime('%Y-%m-%d')}. "
    "Your job is to look up order data and perform calculations to answer user questions. "
    "Guidelines:\n"
    "1. Use `lookup_order` whenever you need to find item details, price, purchase date, or warranty duration.\n"
    "2. Use `calculate` to evaluate any and all mathematical or pricing arithmetic. Do not estimate math.\n"
    "3. If an order cannot be found in the system, explicitly state that you cannot find the order. "
    "Do not hallucinate or invent order data under any circumstances."
)

# Bundle tools and configuration together
config = types.GenerateContentConfig(
    system_instruction=agent_instructions,
    tools=[lookup_order, calculate],
    temperature=0.0 # Keep it deterministic for reliable reasoning chains
)

# ==========================================
# 3. EXECUTION & TRACE CAPTURE
# ==========================================
def run_agent_goal(prompt_text: str):
    print(f"\n🚀 Starting Agent Loop with Goal: '{prompt_text}'")
    print("=" * 60)
    
    # Create a chat session to let the agent manage the loop
    chat = client.chats.create(model="gemini-2.5-flash", config=config)
    
    # Send message to trigger the loop
    response = chat.send_message(prompt_text)
    
    # Read the full historic trace
    history = chat.get_history()
    
    step_counter = 1
    for message in history:
        # Check if the message has parts to read
        if not message.parts:
            continue
            
        # Case 1: The Model's turn (Reasoning / Deciding to call a tool)
        if message.role == "model":
            for part in message.parts:
                # Check if this part is a tool call request
                if hasattr(part, 'function_call') and part.function_call:
                    print(f"\n[Step {step_counter}] 🤖 AGENT REASONING & ACTIONS:")
                    print(f"  -> Decided to call tool: {part.function_call.name}")
                    print(f"     With arguments: {part.function_call.args}")
                    step_counter += 1
                    
        # Case 2: The User/Environment turn (Providing the tool's output back to the model)
        elif message.role == "user":
            for part in message.parts:
                # Check if this part contains a tool response payload
                if hasattr(part, 'function_response') and part.function_response:
                    print(f"[Step {step_counter}] 🔌 TOOL OBSERVED OUTPUT:")
                    # Using .response here extracts the JSON dict payload cleanly
                    print(f"  <- Tool Output: {part.function_response.response}")
                    step_counter += 1

    print("\n" + "=" * 60)
    print("🏁 FINAL AGENT RESPONSE:")
    print(response.text)
    print("=" * 60 + "\n")

if __name__ == "__main__":
    # --- Test Case 1: Multi-Step Goal ---
    multi_step_goal = "I'm thinking of buying two more of order A1001. What would those two cost, and is the original still under warranty?"
    run_agent_goal(multi_step_goal)
    
    # --- Test Case 2: Stretch Goal (Graceful Failure) ---
    stretch_goal = "I want to check order A9999. Can you tell me if it is still under warranty?"
    run_agent_goal(stretch_goal)


🚀 Starting Agent Loop with Goal: 'I'm thinking of buying two more of order A1001. What would those two cost, and is the original still under warranty?'

[Step 1] 🤖 AGENT REASONING & ACTIONS:
  -> Decided to call tool: lookup_order
     With arguments: {'order_id': 'A1001'}
[Step 2] 🔌 TOOL OBSERVED OUTPUT:
  <- Tool Output: {'result': '{"item": "laptop", "price": 1200, "purchased": "2026-05-20", "warranty_months": 12}'}

[Step 3] 🤖 AGENT REASONING & ACTIONS:
  -> Decided to call tool: calculate
     With arguments: {'expression': '1200 * 2'}
[Step 4] 🔌 TOOL OBSERVED OUTPUT:
  <- Tool Output: {'result': '2400'}

🏁 FINAL AGENT RESPONSE:
Two more of order A1001 would cost 2400. The original order is still under warranty, as it was purchased on 2026-05-20 with a 12-month warranty, and today's date is 2026-06-25.


🚀 Starting Agent Loop with Goal: 'I want to check order A9999. Can you tell me if it is still under warranty?'

[Step 1] 🤖 AGENT REASONING & ACTIONS:
  -> Decided to call tool: l